# exp099_pf_multi_observation_likelihood_probe train

Target-free multi-observation likelihood probe for existing PF/Beam/likelihood-PF candidates. This notebook also saves an exp072-style wide train feature cache for downstream ranker experiments.


## Contents

1. Setup and configuration
2. Input and audit/cache contract
3. Run multi-observation likelihood probe and feature cache
4. Preview outputs
5. Metrics and next branch


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from pf_multi_observation_likelihood_probe import run_from_config, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

if DEBUG:
    config.setdefault("audit", {})["max_rows"] = int(os.environ.get("EXPERIMENT_MAX_ROWS", "20000"))
    config.setdefault("audit", {})["save_candidate_long"] = False

print(json.dumps({
    "experiment": EXPERIMENT_NAME,
    "route": get_nested(config, "experiment.route"),
    "status": get_nested(config, "experiment.status"),
    "parent": get_nested(config, "lineage.parent"),
    "cache_parent": get_nested(config, "lineage.cache_parent"),
    "mode": get_nested(config, "audit.mode"),
    "debug": DEBUG,
    "max_rows": get_nested(config, "audit.max_rows"),
    "artifacts_dir": str(paths.artifacts_dir),
}, indent=2, sort_keys=True))


## 2. Input and audit/cache contract


In [ ]:
print(json.dumps({
    "multi_observation_likelihood": get_nested(config, "model.multi_observation_likelihood"),
    "candidate_sets": get_nested(config, "audit.candidate_sets"),
    "thresholds_ft": get_nested(config, "audit.thresholds_ft"),
    "topk_values": get_nested(config, "audit.topk_values"),
    "leakage_policy": get_nested(config, "validation.leakage_policy"),
    "expected_train_artifacts": get_nested(config, "audit.expected_train_artifacts"),
}, indent=2, ensure_ascii=False))

for label, dotted in {
    "exp072_train_feature_cache_local": "data.exp072_train_feature_cache_local",
    "train_dir": "data.train_dir",
}.items():
    value = get_nested(config, dotted)
    path = Path(value) if value else None
    print(label, value, "exists=" + str(path.exists() if path else False))


## 3. Run multi-observation likelihood probe and feature cache


In [ ]:
summary = run_from_config(config)
print(json.dumps(to_jsonable({
    "status": summary["status"],
    "runtime_seconds": summary["runtime_seconds"],
    "rows": summary["source"]["rows"],
    "wells": summary["source"]["wells"],
    "generated_candidates": summary["multi_observation_likelihood"]["generated_candidates"],
    "train_feature_cache": {
        "variant": summary["train_feature_cache"]["variant"],
        "feature_count": summary["train_feature_cache"]["feature_count"],
        "outputs": summary["train_feature_cache"]["outputs"],
    },
    "recommendation": summary["probe_decision"]["recommendation"],
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
}), indent=2, sort_keys=True))


## 4. Preview outputs


In [ ]:
artifact_paths = {name: paths.artifacts_dir / filename for name, filename in summary["outputs"].items() if filename}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

def preview_csv(name: str, n: int = 10) -> pd.DataFrame:
    path = artifact_paths[name]
    frame = pd.read_csv(path, nrows=n)
    display(frame)
    return frame

candidate_metrics_preview = preview_csv("candidate_metrics")
rank_metrics_preview = preview_csv("rank_metrics")
bucket_metrics_preview = preview_csv("bucket_metrics")
multiobs_well_summary_preview = preview_csv("multiobs_well_summary")


## 5. Metrics and next branch


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "updated_at": datetime.now(UTC).isoformat(),
    "route": get_nested(config, "experiment.route"),
    "metric": "multi_observation_likelihood_candidate_audit",
    "source": summary["source"],
    "multi_observation_likelihood": summary["multi_observation_likelihood"],
    "train_feature_cache": summary["train_feature_cache"],
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
    "probe_decision": summary["probe_decision"],
    "outputs": summary["outputs"],
}
metrics_path = paths.experiment_dir / "metrics.json"
with metrics_path.open("w") as fp:
    json.dump(to_jsonable(metrics), fp, indent=2, sort_keys=True)
print(json.dumps(to_jsonable(metrics), indent=2, sort_keys=True))
